# climakitae usage Summary for JTree

This notebook consolidates the climate data analysis for Joshua Tree National Park. It explicitly fetches data using `climakitae`, processes it (masking, spatial averaging, anomaly calculation), runs diagnostics, and produces visualizations. Also again remmeber that using this workflow with 3 environments (base/py-env/R) to register python notebooks, like this one with the py-env kernel (selection box in the top right corner), and R notebooks, with the R kernel.

**Key Steps:**
1.  **Configuration**: Set scenarios, variables, and periods.
2.  **Data Retrieval**: Explicitly call `ck.get_data`.
3.  **Processing**: Mask to park boundary, calculate spatial averages, and compute anomalies.
4.  **Diagnostics**: Check spatial coverage and data quality.
5.  **Visualization**: Plot time series (Spread & Distribution). 
6.  **Export**: Save processed data for R.


In [1]:
import climakitae as ck
from climakitae.core.data_interface import get_data
import xarray as xr
import numpy as np
import pandas as pd
import geopandas as gpd
import rioxarray as rxr
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")
xr.set_options(keep_attrs=True)

# Graphics settings
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Configuration & Helper Functions

Define the scenarios, variables, and boundaries. We also define the `processVariable` function here, which encapsulates the entire data fetching and processing pipeline.

In [ ]:
# --- CONFIGURATION ---

# Constants
SCENARIOS = [
    "Historical Climate", 
    "SSP 2-4.5", 
    "SSP 3-7.0", 
    "SSP 5-8.5"
]
RES = "3 km"
# UPDATED: User uses Statistical downscaling for 3km resolution availability
DOWNSCALING = "Statistical"

VARIABLES = {
    "T_Avg": "Air Temperature at 2m",    # Will be overridden if DOWNSCALING = "Statistical"
    "Precip": "Precipitation (total)"
}

# Time Periods
BASELINE_YEARS = (1991, 2020)
FUTURE_YEARS = (2015, 2100) # Full future range
TIMESPAN = (1980, 2100)

# Shapefile Path
SHP_PATH = "../JoshuaTreeOutlines/JoshuaTree/Joshua_Tree_National_Park.shp"
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load Boundary
try:
    jTreeBoundary = gpd.read_file(SHP_PATH)
    # Reproject to WGS84 (EPSG:4326) which acts as the standard
    jTreeBoundary_WGS84 = jTreeBoundary.to_crs("EPSG:4326")
    bounds = jTreeBoundary_WGS84.total_bounds
    latitidualSlice = (bounds[1], bounds[3])
    longitudinalSlice = (bounds[0], bounds[2])
    print(f"Loaded boundary. Bounds: {bounds}")
except Exception as e:
    print(f"Error loading shapefile: {e}. Using default box.")
    # Fallback (approximate Joshua Tree box)
    latitidualSlice = (33.5, 34.2)
    longitudinalSlice = (-116.3, -115.0)
    jTreeBoundary_WGS84 = None

In [ ]:
# --- PROCESSING FUNCTION ---

def processVariable(key, varName, scenarios, resolution, timespan, latSlice, lonSlice, boundaryGdf):
    """
    Fetches, processes, and returns time series data for a given variable.
    Includes explicit ck.get_data call.
    """
    print(f"\n--- Processing {key} ---")
    
    # 1. EXPLICIT DATA FETCHING
    data = None
    try:
        # Handle Logic for Statistical vs Dynamical Variable Names
        if DOWNSCALING == "Statistical" and key == "T_Avg":
            print(f"  Statistical Method selected: Fetching T_Max and T_Min to calculate T_Avg...")
            
            # Fetch Max
            print("    Fetching Daily Maximum Air Temperature...")
            tmax = ck.core.data_interface.get_data(
                variable="Daily Maximum Air Temperature",
                resolution=resolution,
                downscaling_method=DOWNSCALING,
                timescale="monthly",
                scenario=scenarios,
                time_slice=timespan,
                latitude=latSlice,
                longitude=lonSlice
            )
            
            # Fetch Min
            print("    Fetching Daily Minimum Air Temperature...")
            tmin = ck.core.data_interface.get_data(
                variable="Daily Minimum Air Temperature",
                resolution=resolution,
                downscaling_method=DOWNSCALING,
                timescale="monthly",
                scenario=scenarios,
                time_slice=timespan,
                latitude=latSlice,
                longitude=lonSlice
            )
            
            # Average them
            print("    Calculating Average (T_Max + T_Min) / 2 ...")
            data = (tmax + tmin) / 2
            # Inherit attributes from tmax (units should be same)
            data.attrs = tmax.attrs
            
        else:
            # Standard Fetch (Dynamical or Precip)
            target_var = varName
            if DOWNSCALING == "Statistical" and key == "Precip":
                 target_var = "Precipitation (total)"
            
            print(f"  Fetching {target_var}...")
            data = ck.core.data_interface.get_data(
                variable=target_var,
                resolution=resolution,
                downscaling_method=DOWNSCALING,
                timescale="monthly",
                scenario=scenarios,
                time_slice=timespan,
                latitude=latSlice,
                longitude=lonSlice
            )
            
        print(f"    Data shape: {data.shape}")

    except Exception as e:
        print(f"    Error fetching data: {e}")
        return None

    # 2. Unit Conversion
    if key == "Precip":
        # Convert kg/m2/s to mm/month (approximate)
        # Actually, usually climakitae returns mm or flux. Let's assume standard handling.
        # If units are kg m-2 s-1, convert.
        if data.attrs.get('units') == "kg m-2 s-1":
            print("    Converting Precip units from kg m-2 s-1 to mm/month...")
            # method: multiply by seconds in month. Simplified: * 2628000 (avg seconds)
            # Better: use proper Calendar handling if possible, or simple scalar.
            # Using 86400 * 30.44 as approx (2630016)
            data = data * 2630016
            data.attrs['units'] = "mm/month"
    elif key == "T_Avg":
        if data.attrs.get('units') == "K":
            print("    Converting Temp from Kelvin to Celsius...")
            data = data - 273.15
            data.attrs['units'] = "degC"

    # 3. Spatial Masking & Weighting
    if boundaryGdf is not None:
        print("  Masking data to park boundary...")
        # Ensure CRS match
        if hasattr(data, 'rio'):
             data = data.rio.write_crs("EPSG:4326")
        
        try:
            # Clip
            masked = data.rio.clip(boundaryGdf.geometry, boundaryGdf.crs, drop=False)
            
            # Calculate Weights (Diagnostic)
            # NOTE: We use cosine(latitude) weighting to account for the convergence of meridians 
            # (i.e., the "curvature of the earth") on a WGS84 grid. This avoids the need to 
            # reproject to an equal-area projection like Albers for this aggregation.
            print("    Calculating spatial weights (accounting for earth curvature)...")
            weights = np.cos(np.deg2rad(masked.y))
            weights.name = "weights"

            # --- DIAGNOSTIC PLOT ---
            try:
                print("    Visualizing spatial weights...")
                plt.figure(figsize=(5, 4))
                weights.plot()
                plt.title(f"Spatial Weights (Cosine Lat) - {key}")
                plt.axis('off')
                plt.show()
            except Exception as e:
                print(f"    (Skipping weight plot: {e})")
            # -----------------------
            
            # Apply weighted mean
            spatial_avg = masked.weighted(weights).mean(dim=["x", "y"], skipna=True)
        except Exception as e:
            print(f"    Masking/Weighting failed: {e}. Using simple mean.")
            spatial_avg = data.mean(dim=["x", "y"], skipna=True)
    else:
        spatial_avg = data.mean(dim=["x", "y"], skipna=True)

    # 4. Annual Aggregation
    print("  Aggregating to annual...")
    if key == "Precip":
        annual = spatial_avg.resample(time="YE").sum()
        annual.attrs['units'] = "mm/year"
    else:
        annual = spatial_avg.resample(time="YE").mean()
        annual.attrs['units'] = "degC"

    # 5. Anomaly Calculation
    print("  Calculating anomalies...")
    try:
        baseline_slice = annual.sel(time=slice(str(BASELINE_YEARS[0]), str(BASELINE_YEARS[1])))
        baseline_mean = baseline_slice.mean(dim="time")
        
        if key == "Precip":
            # Percent change, exclude low baselines
            avg_base = baseline_mean.mean().item()
            if abs(avg_base) < 1.0:
                print("    Baseline precip < 1mm. Using absolute delta.")
                anom = annual - baseline_mean
                anom.attrs['units'] = "Delta mm/year"
            else:
                anom = ((annual - baseline_mean) / baseline_mean) * 100
                anom.attrs['units'] = "% Change"
        else:
            anom = annual - baseline_mean
            anom.attrs['units'] = "Delta degC"
            
    except Exception as e:
        print(f"    Anomaly calculation error: {e}")
        return None

    # 6. Smoothing
    print("  Applying 10-year smoothing...")
    smoothed = anom.rolling(time=10, center=True, min_periods=1).mean()
    
    # Return result
    return smoothed


## 2. Visualization Functions

These functions duplicate the logic from `pythonForTimeseries.ipynb` to provide immediate feedback on the processed data.

In [ ]:
def plotAnnualSpread(dataArray, varKey):
    """
    Plots annual anomalies with individual simulations and ensemble means.
    """
    print(f"  Plotting Spread for {varKey}...")
    df = dataArray.to_dataframe(name='Anomaly').reset_index()
    
    # Cleanup
    df['Year'] = df['time'].dt.year
    if 'scenario' in df.columns:
        # Clean scenario names (remove 'Historical + ' prefix)
        df['Scenario'] = df['scenario'].astype(str).str.replace("Historical + ", "", regex=False)
        # Ensure Historical is labeled correctly for years <= 2014 if accidentally merged
        if 2014 in df['Year'].values:
             df.loc[df['Year'] <= 2014, 'Scenario'] = "Historical Climate"
    else:
        df['Scenario'] = "Unknown"
        
    plt.figure(figsize=(12, 6))
    
    # Plot Individual Simulations (Light)
    sns.lineplot(
        data=df, x='Year', y='Anomaly', hue='Scenario', units='simulation', estimator=None,
        lw=0.5, alpha=0.3, palette={'Historical Climate': 'grey', 'SSP 2-4.5': '#377eb8', 'SSP 3-7.0': '#D55E00', 'SSP 5-8.5': '#840312'}
    )
    
    # Plot Ensemble Means (Heavy)
    sns.lineplot(
        data=df, x='Year', y='Anomaly', hue='Scenario', 
        lw=3, palette={'Historical Climate': 'black', 'SSP 2-4.5': '#377eb8', 'SSP 3-7.0': '#D55E00', 'SSP 5-8.5': '#840312'}
    )
    
    plt.title(f"{varKey}: Annual Anomalies (Relative to {BASELINE_YEARS[0]}-{BASELINE_YEARS[1]})")
    plt.ylabel(dataArray.attrs.get('units', 'Anomaly'))
    plt.xlabel("Year")
    plt.xlim(1980, 2100)
    plt.grid(True, alpha=0.3)
    plt.show()

## 3. Execution Loop

Run the processing for each variable, generate plots, and collect data for export.

In [ ]:
timeSeriesData = {}

for key, varName in VARIABLES.items():
    # CALL PROCESS VARIABLE
    tsData = processVariable(
        key, 
        varName, 
        SCENARIOS, 
        RES, 
        TIMESPAN, 
        latitidualSlice, 
        longitudinalSlice, 
        jTreeBoundary_WGS84
    )
    
    if tsData is not None:
        timeSeriesData[key] = tsData
        # PLOT IMMEDIATELY
        plotAnnualSpread(tsData, key)

## 4. Export to CSV

Format the data for compatibility with the R visualization notebook.

In [ ]:
all_dfs = []

for key, data in timeSeriesData.items():
    df = data.to_dataframe(name='Anomaly').reset_index()
    df['Variable'] = key
    df['Year'] = df['time'].dt.year
    df = df.drop(columns=['time'], errors='ignore')
    
    # Standardize Scenario Names
    if 'scenario' in df.columns:
        df['Scenario'] = df['scenario'].astype(str).str.replace("Historical + ", "", regex=False)
        df.loc[df['Year'] <= 2014, 'Scenario'] = "Historical Climate"
        df = df.drop(columns=['scenario'])
        
    if 'simulation' in df.columns:
        df = df.rename(columns={'simulation': 'Simulation'})
    
    # Drop NaNs/Infs
    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=['Anomaly'])
    
    all_dfs.append(df)

if all_dfs:
    final_df = pd.concat(all_dfs)
    # Select columns
    cols = ['Year', 'Scenario', 'Simulation', 'Variable', 'Anomaly']
    final_df = final_df[[c for c in cols if c in final_df.columns]]
    
    out_file = os.path.join(OUTPUT_DIR, "processed_climate_data.csv")
    final_df.to_csv(out_file, index=False)
    print(f"\nSuccess! Saved processed data to {out_file}")
    print(final_df.head())
else:
    print("No data to save.")

## Appendix: Common Pitfalls & Notes

1.  **Variable Names depend on Method**:
    *   **Dynamical**: Uses `Air Temperature at 2m`.
    *   **Statistical**: Uses `Daily Maximum Air Temperature` and `Daily Minimum Air Temperature`. You must fetch both and average them to get `T_Avg`.
2.  **Resolution Availability**:
    *   `3 km` resolution is widely available for **Statistical** downscaling across all SSPs.
    *   **Dynamical** downscaling often has gaps for certain SSPs at high resolution.
3.  **Units**:
    *   Precipitation often comes as `kg/m2/s` (flux). It must be converted to `mm/month` or `mm/year`.
4.  **Scenario Naming**:
    *   Raw data often labels scenarios like `"Historical + SSP 3-7.0"`. Logic is needed to strip the prefix and handle the historical period correctly.
5.  **Low Baseline Precipitation**:
    *   In deserts, baseline precip can be near zero. Calculating `% Change` leads to massive spikes (e.g., +1000%). Using **Absolute Difference** (`Delta mm/year`) is safer when baseline < 1mm.
6.  **Area Weighting (Earth Curvature)**:
    *   When working with a WGS84 grid (lat/lon), pixel areas get smaller as you move from the equator to the poles due to the convergence of meridians.
    *   We use `weights = np.cos(np.deg2rad(lat))` to account for this. This is the standard method for area-weighted spatial averaging in this coordinate system, avoiding the need for reprojection to an equal-area CRS (like Albers).

7.  **Why this "Kernel & Environment" System is Helpful & Nice**:
    *   **The Problem**: Python is amazing for heavy data processing (libraries like `xarray` and `dask` handle terabytes of climate data effortlessly). R is amazing for publication-quality visualization (`ggplot2` is often considered superior for static plots). We want to use *both*.
    *   **The Architecture**: This environment (Jupyter Hub/Lab) allows you to switch between Python and R kernels effortlessly. However, because they are separate processes, they **do not share memory**.
        *   **Best of Both Worlds**: We solve this by using the shared filesystem. This notebook (Python) does the heavy lifting: fetching, cleaning, and aggregating the massive data down to a small, manageable size. It saves this clean data to a CSV (`outputs/processed_climate_data.csv`).
        *   **The Handoff**: You then open the R notebook (`02_Climate_Data_Visualization.ipynb`), which simply reads that CSV and focuses purely on making beautiful plots. This separation of concerns makes your workflow robust, reproducible, and cleaner.
    *   **Dask & Memory**: Under the hood, Python uses **Dask** for "lazy loading". Data isn't loaded into RAM until you ask for it (e.g., `.plot()` or `.to_csv()`). This is why specific steps might take a moment to run—they are triggering the actual computation graph.
